# Analisis Kualitas Udara Kabupaten Sampang

**Pertemuan 1 — Business Understanding, Data Understanding, Data Collection, dan Eksplorasi Awal**

Notebook ini mendokumentasikan proses pengamatan kualitas udara di Kabupaten Sampang menggunakan data Sentinel-5P/TROPOMI dari Copernicus Data Space Ecosystem. Fokus pengamatan adalah parameter **Nitrogen Dioxide (NO₂)** dengan periode permintaan data **24 Agustus 2025 sampai 24 Agustus 2026**.


# 1. Business Understanding

Kualitas udara menggambarkan kondisi udara berdasarkan keberadaan berbagai polutan yang dapat memengaruhi manusia dan lingkungan. Salah satu cara menyampaikan kondisi kualitas udara adalah **Air Quality Index (AQI)**, yaitu indeks yang mengubah informasi konsentrasi polutan menjadi kategori yang lebih mudah dipahami.

Polutan yang umum diamati antara lain **NO₂, CO, O₃, SO₂, PM₂.₅, dan PM₁₀**. Pada proyek ini pengamatan difokuskan pada **NO₂** karena band tersebut tersedia pada collection Sentinel-5P yang digunakan.

Tujuan tahap ini adalah memperoleh data NO₂ wilayah Sampang, memahami kelengkapan data, serta melakukan eksplorasi awal terhadap distribusi dan perubahan nilai NO₂.


# 2. Data Understanding

Data diperoleh dari **Copernicus Data Space Ecosystem** melalui collection **`SENTINEL_5P_L2`**.

Berdasarkan metadata yang diperiksa pada notebook sumber, collection tersebut menyediakan beberapa parameter atmosfer, termasuk CO, HCHO, NO₂, O₃, SO₂, CH₄, aerosol index, informasi awan, dan `dataMask`.

| Komponen | Keterangan |
|---|---|
| Sumber | Copernicus Data Space Ecosystem |
| Platform | Sentinel-5 Precursor |
| Instrumen | TROPOMI |
| Collection | `SENTINEL_5P_L2` |
| Parameter | NO₂ |
| Wilayah | AOI pengamatan Sampang |
| Periode permintaan | 24-08-2025 s.d. 24-08-2026 |
| Resolusi waktu | Harian setelah agregasi |
| Format hasil | NetCDF |

**Catatan:** AOI yang digunakan pada collecting adalah bounding box/polygon dengan koordinat barat 113.0, selatan -7.5, timur 113.6, dan utara -6.9. AOI ini tidak diklaim sebagai batas administratif presisi Kabupaten Sampang.


# 3. Data Collection

Pengambilan data dilakukan menggunakan **openEO** untuk mengakses Copernicus Data Space Ecosystem.

Tahap collecting terdiri dari:

1. Instalasi library.
2. Membuat koneksi Copernicus.
3. Autentikasi.
4. Mencari collection Sentinel-5P.
5. Memeriksa metadata collection.
6. Memilih band NO₂.
7. Menentukan AOI dan periode.
8. Agregasi temporal harian.
9. Agregasi spasial.
10. Menjalankan batch job.
11. Mengunduh hasil NetCDF.


### Penjelasan
Library dipasang untuk proses akses Copernicus, pengolahan NetCDF, analisis data, dan visualisasi.


In [ ]:
!pip install openeo xarray netcdf4 pandas numpy matplotlib folium

: 

### Penjelasan
Kode membuat koneksi ke layanan openEO pada Copernicus Data Space Ecosystem.


In [ ]:
import openeo

connection = openeo.connect(
    "https://openeo.dataspace.copernicus.eu"
)

print(connection)

### Penjelasan
Kode melakukan autentikasi OIDC agar notebook dapat mengakses layanan Copernicus menggunakan akun pengguna.


In [ ]:
connection.authenticate_oidc()

### Penjelasan
Kode mengambil daftar collection yang tersedia dan mencari collection Sentinel-5P.


In [ ]:
collections = connection.list_collection_ids()

print("Jumlah koleksi:", len(collections))

for collection in collections:
    if "5P" in collection.upper():
        print(collection)

### Penjelasan
Metadata collection diperiksa untuk mengetahui band, dimensi, resolusi, instrumen, dan informasi dataset.


In [ ]:
info = connection.describe_collection("SENTINEL_5P_L2")

print(info)

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
print(info.get("cube:dimensions"))

## 3.1 Menentukan AOI, Periode, dan Memuat Data NO₂


### Penjelasan
Kode memuat collection `SENTINEL_5P_L2`, membatasi AOI dan periode pengamatan, serta memilih band `NO2`.


In [ ]:
# ============================================
# KODE 5 - LOAD SENTINEL-5P NO2 SAMPANG
# ============================================

spatial_extent = {
    "west": 113.0,
    "south": -7.5,
    "east": 113.6,
    "north": -6.9,
    "crs": "EPSG:4326"
}

temporal_extent = [
    "2025-08-24",
    "2026-08-24"
]

no2_cube = connection.load_collection(
    "SENTINEL_5P_L2",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["NO2"]
)

print("Data NO2 berhasil dimuat.")
print(no2_cube)

### Penjelasan
Data NO₂ dirata-ratakan untuk setiap periode satu hari sehingga membentuk deret waktu harian.


In [ ]:
# ============================================
# KODE 6 - AGREGASI NO2 PER HARI
# ============================================

no2_daily = no2_cube.aggregate_temporal_period(
    reducer="mean",
    period="day"
)

print(no2_daily)

### Penjelasan
Kode membuat polygon AOI berdasarkan koordinat yang digunakan dalam proses pengambilan data.


In [ ]:
# ============================================
# KODE 7 - AOI SAMPANG SEMENTARA
# ============================================

aoi_sampang = {
    "type": "Polygon",
    "coordinates": [[
        [113.0, -7.5],
        [113.6, -7.5],
        [113.6, -6.9],
        [113.0, -6.9],
        [113.0, -7.5]
    ]]
}

print("AOI berhasil dibuat.")

### Penjelasan
Kode membuat polygon AOI berdasarkan koordinat yang digunakan dalam proses pengambilan data.


In [ ]:
# ============================================
# KODE 8 - AGREGASI SPASIAL SAMPANG
# ============================================

no2_sampang = no2_daily.aggregate_spatial(
    geometries=aoi_sampang,
    reducer="mean"
)

print(no2_sampang)

## 3.2 Menjalankan Batch Job dan Mengambil Hasil


### Penjelasan
Batch job dibuat untuk menjalankan proses pengolahan data pada backend openEO dan menyimpan hasil dalam format NetCDF.


In [ ]:
# ============================================
# KODE 9 - BUAT BATCH JOB
# ============================================

job = no2_sampang.create_job(
    title="NO2 Kabupaten Sampang 2025-2026",
    out_format="NetCDF"
)

print("Job dibuat:", job.job_id)

### Penjelasan
Batch job dijalankan dan notebook menunggu sampai proses selesai.


In [ ]:
job.start_and_wait()

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
# ============================================
# KODE 10 - AMBIL HASIL JOB
# ============================================

results = job.get_results()

print(results)


### Penjelasan
Hasil job diunduh dan disimpan ke folder `data_no2_sampang`.


In [ ]:
# ============================================
# KODE 11 - DOWNLOAD HASIL
# ============================================

import os

os.makedirs("./data_no2_sampang", exist_ok=True)

results.download_files(
    target="./data_no2_sampang"
)

print("Download selesai.")

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
# ============================================
# KODE 12 - CEK FILE HASIL
# ============================================

for root, dirs, files in os.walk("./data_no2_sampang"):
    for file in files:
        print(os.path.join(root, file))

## 4. Data Preparation


### Penjelasan
File NetCDF dibaca menggunakan xarray untuk memeriksa struktur dataset dan variabel NO₂.


In [ ]:
# ============================================
# KODE 13 - MEMBACA NETCDF
# ============================================

import xarray as xr

nc_file = "./data_no2_sampang/timeseries.nc"

ds = xr.open_dataset(nc_file)

print(ds)

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
# ============================================
# KODE 14 - CEK VARIABEL DATA
# ============================================

print("Variables:")
print(list(ds.data_vars))

print("\nDimensions:")
print(ds.dims)

print("\nCoordinates:")
print(list(ds.coords))

### Penjelasan
Dataset ditampilkan dalam bentuk tabel untuk melihat contoh data hasil pengamatan.


In [ ]:
# ============================================
# KODE 15 - LIHAT DATA NO2
# ============================================

print(ds.to_dataframe().head(20))

### Penjelasan
Data xarray diubah menjadi DataFrame, kolom yang diperlukan dipilih, tanggal diubah ke tipe datetime, dan data diurutkan.


In [ ]:
# ============================================
# KODE 16-17 - MEMBUAT DAN MEMBERSIHKAN DATAFRAME
# ============================================

import pandas as pd

# Membaca hasil NetCDF
df = ds.to_dataframe().reset_index()

# Pilih kolom yang diperlukan
df = df[
    ["t", "lat", "lon", "NO2"]
].copy()

# Rename kolom
df = df.rename(columns={
    "t": "tanggal"
})

# Pastikan tanggal bertipe datetime
df["tanggal"] = pd.to_datetime(df["tanggal"])

# Urutkan berdasarkan tanggal
df = df.sort_values("tanggal").reset_index(drop=True)

print("Data berhasil disiapkan.")
print("\n5 data pertama:")
print(df.head())

print("\n5 data terakhir:")
print(df.tail())

print("\nInformasi dataset:")
print(df.info())

## 5. Pemeriksaan Missing Values dan Missing Dates


### Penjelasan
Pemeriksaan ini digunakan untuk mengetahui jumlah dan persentase missing value pada setiap kolom.


In [ ]:
# ============================================
# KODE 18 - CEK MISSING VALUE
# ============================================

print("Jumlah missing value:")
print(df.isnull().sum())

print("\nPersentase missing value:")
print(
    (df.isnull().sum() / len(df) * 100).round(2)
)

### Penjelasan
Kode memeriksa tanggal pengamatan paling awal, paling akhir, dan jumlah observasi.


In [ ]:
# ============================================
# KODE 19 - CEK RENTANG DATA
# ============================================

print("Tanggal paling awal :", df["tanggal"].min())
print("Tanggal paling akhir:", df["tanggal"].max())

print("Jumlah observasi    :", len(df))

### Penjelasan
Rentang tanggal lengkap dibandingkan dengan tanggal yang tersedia untuk menemukan tanggal yang hilang.


In [ ]:
# ============================================
# KODE 20 - CEK TANGGAL YANG HILANG
# ============================================

tanggal_lengkap = pd.date_range(
    start="2025-08-24",
    end="2026-08-24",
    freq="D"
)

tanggal_data = pd.DatetimeIndex(
    df["tanggal"].dt.normalize()
)

tanggal_hilang = tanggal_lengkap.difference(tanggal_data)

print("Jumlah tanggal seharusnya :", len(tanggal_lengkap))
print("Jumlah tanggal tersedia   :", len(tanggal_data))
print("Jumlah tanggal hilang     :", len(tanggal_hilang))

if len(tanggal_hilang) > 0:
    print("\nTanggal yang hilang:")
    for tanggal in tanggal_hilang:
        print(tanggal.strftime("%Y-%m-%d"))
else:
    print("\nTidak ada tanggal yang hilang.")

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
# ============================================
# KODE 21 - STATUS KELENGKAPAN DATA
# ============================================

tanggal_lengkap = pd.DataFrame({
    "tanggal": pd.date_range(
        start="2025-08-24",
        end="2026-08-24",
        freq="D"
    )
})

# Tandai tanggal yang mempunyai data
tanggal_lengkap["tersedia"] = (
    tanggal_lengkap["tanggal"].isin(df["tanggal"])
)

print(tanggal_lengkap.head(10))

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
print(tanggal_lengkap["tersedia"].value_counts())


### Penjelasan
Persentase kelengkapan dihitung dari jumlah hari yang memiliki data dibandingkan dengan total hari.


In [ ]:
# ============================================
# KODE 22 - PERSENTASE KELENGKAPAN DATA
# ============================================

total_hari = len(tanggal_lengkap)
hari_tersedia = tanggal_lengkap["tersedia"].sum()
hari_hilang = total_hari - hari_tersedia

persentase_tersedia = hari_tersedia / total_hari * 100
persentase_hilang = hari_hilang / total_hari * 100

print(f"Total hari        : {total_hari}")
print(f"Hari tersedia     : {hari_tersedia}")
print(f"Hari tidak tersedia: {hari_hilang}")
print(f"Data tersedia     : {persentase_tersedia:.2f}%")
print(f"Data tidak tersedia: {persentase_hilang:.2f}%")

## 6. Statistik Deskriptif NO₂


### Penjelasan
Statistik deskriptif digunakan untuk melihat count, mean, standar deviasi, minimum, kuartil, median, dan maksimum NO₂.


In [ ]:
# ============================================
# KODE 23 - STATISTIK DESKRIPTIF NO2
# ============================================

print(df["NO2"].describe())

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
# ============================================
# KODE 24 - STATISTIK LENGKAP
# ============================================

statistik_no2 = pd.DataFrame({
    "Minimum": [df["NO2"].min()],
    "Maximum": [df["NO2"].max()],
    "Mean": [df["NO2"].mean()],
    "Median": [df["NO2"].median()],
    "Std": [df["NO2"].std()],
    "Q1": [df["NO2"].quantile(0.25)],
    "Q3": [df["NO2"].quantile(0.75)]
})

print(statistik_no2)

### Penjelasan
Kode mengambil lima nilai NO₂ tertinggi dan lima nilai terendah untuk melihat pengamatan ekstrem.


In [ ]:
# ============================================
# KODE 25 - NILAI TERTINGGI DAN TERENDAH
# ============================================

print("5 nilai NO2 tertinggi:")
print(
    df.nlargest(5, "NO2")[
        ["tanggal", "NO2"]
    ]
)

print("\n5 nilai NO2 terendah:")
print(
    df.nsmallest(5, "NO2")[
        ["tanggal", "NO2"]
    ]
)

## 7. Identifikasi Outlier


### Penjelasan
Metode IQR digunakan untuk menentukan batas bawah dan batas atas dalam identifikasi outlier.


In [ ]:
# ============================================
# KODE 26 - IDENTIFIKASI OUTLIER DENGAN IQR
# ============================================

Q1 = df["NO2"].quantile(0.25)
Q3 = df["NO2"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1          :", Q1)
print("Q3          :", Q3)
print("IQR         :", IQR)
print("Lower Bound :", lower_bound)
print("Upper Bound :", upper_bound)

### Penjelasan
Metode IQR digunakan untuk menentukan batas bawah dan batas atas dalam identifikasi outlier.


In [ ]:
# ============================================
# KODE 27 - MENANDAI OUTLIER
# ============================================

df["status_outlier"] = "Normal"

df.loc[
    (df["NO2"] < lower_bound) |
    (df["NO2"] > upper_bound),
    "status_outlier"
] = "Outlier"

print(df["status_outlier"].value_counts())

### Penjelasan
Setiap observasi diberi status Normal atau Outlier berdasarkan batas IQR.


In [ ]:
# ============================================
# KODE 28 - DAFTAR OUTLIER
# ============================================

outliers = df[
    df["status_outlier"] == "Outlier"
][
    ["tanggal", "NO2", "status_outlier"]
]

print(outliers.to_string(index=False))

### Penjelasan
Kode berikut merupakan bagian dari proses pengolahan dan analisis data Sampang.


In [ ]:
# ============================================
# KODE 29 - PERSENTASE OUTLIER
# ============================================

jumlah_outlier = len(outliers)
total_data = len(df)

persentase_outlier = (
    jumlah_outlier / total_data * 100
)

print("Total data       :", total_data)
print("Jumlah outlier   :", jumlah_outlier)
print(f"Persentase outlier: {persentase_outlier:.2f}%")

### Penjelasan
Metode IQR digunakan untuk menentukan batas bawah dan batas atas dalam identifikasi outlier.


In [ ]:
# ============================================
# KODE 30 - KLASIFIKASI OUTLIER
# ============================================

df["jenis_outlier"] = "Normal"

df.loc[
    df["NO2"] > upper_bound,
    "jenis_outlier"
] = "Outlier Tinggi"

df.loc[
    df["NO2"] < lower_bound,
    "jenis_outlier"
] = "Outlier Rendah"

print(df["jenis_outlier"].value_counts())

### Penjelasan
Outlier dibedakan menjadi Outlier Tinggi dan Outlier Rendah berdasarkan posisinya terhadap batas IQR.


In [ ]:
# ============================================
# KODE 31 - TABEL OUTLIER
# ============================================

tabel_outlier = df[
    df["jenis_outlier"] != "Normal"
][
    ["tanggal", "NO2", "jenis_outlier"]
].copy()

print(tabel_outlier.to_string(index=False))

## 8. Analisis Perubahan NO₂


### Penjelasan
Perubahan harian NO₂ dihitung menggunakan selisih antarobservasi dan persentase perubahan.


In [ ]:
# ============================================
# KODE 32 - PERUBAHAN HARIAN NO2
# ============================================

df["perubahan_NO2"] = df["NO2"].diff()

df["persentase_perubahan"] = (
    df["NO2"].pct_change() * 100
)

print(
    df[
        ["tanggal", "NO2", "perubahan_NO2",
         "persentase_perubahan"]
    ].head(15)
)

### Penjelasan
Kode mengambil lima nilai NO₂ tertinggi dan lima nilai terendah untuk melihat pengamatan ekstrem.


In [ ]:
# ============================================
# KODE 33 - LONJAKAN PERUBAHAN TERBESAR
# ============================================

df["abs_perubahan"] = df["perubahan_NO2"].abs()

lonjakan = df.nlargest(
    15,
    "abs_perubahan"
)[
    [
        "tanggal",
        "NO2",
        "perubahan_NO2",
        "abs_perubahan"
    ]
]

print(lonjakan.to_string(index=False))

## 9. Visualisasi


### Penjelasan
Grafik time series digunakan untuk melihat perubahan nilai NO₂ sepanjang periode pengamatan.


In [ ]:
# ============================================
# KODE 34 - VISUALISASI TIME SERIES NO2
# ============================================

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.plot(
    df["tanggal"],
    df["NO2"],
    marker=".",
    linewidth=1
)

plt.xlabel("Tanggal")
plt.ylabel("NO₂ (mol/m²)")
plt.title(
    "Time Series Konsentrasi NO₂ "
    "Wilayah Pengamatan Sampang"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Penjelasan
Outlier dibedakan menjadi Outlier Tinggi dan Outlier Rendah berdasarkan posisinya terhadap batas IQR.


In [ ]:
# ============================================
# KODE 35 - VISUALISASI OUTLIER
# ============================================

normal = df[
    df["jenis_outlier"] == "Normal"
]

outlier = df[
    df["jenis_outlier"] != "Normal"
]

plt.figure(figsize=(15, 6))

plt.plot(
    df["tanggal"],
    df["NO2"],
    marker=".",
    linewidth=1,
    label="NO₂"
)

plt.scatter(
    outlier["tanggal"],
    outlier["NO2"],
    s=60,
    label="Outlier"
)

plt.xlabel("Tanggal")
plt.ylabel("NO₂ (mol/m²)")
plt.title(
    "Identifikasi Outlier NO₂ "
    "Wilayah Pengamatan Sampang"
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 10. Ringkasan Hasil Pertemuan 1

Berdasarkan output notebook sumber:

- Data yang diminta mencakup periode **24 Agustus 2025–24 Agustus 2026**.
- Hasil akhir memiliki **315 observasi**.
- Tidak terdapat missing value pada kolom `tanggal`, `lat`, `lon`, dan `NO2`.
- Terdapat **51 tanggal tanpa observasi** dari total 366 hari.
- Kelengkapan data adalah **86,07%**.
- Rata-rata NO₂ sekitar **0,000021**.
- Nilai minimum **-0,000008** dan maksimum **0,000054**.
- Dengan metode IQR ditemukan **10 outlier (3,17%)**, terdiri dari **7 outlier tinggi** dan **3 outlier rendah**.

Hasil ini menjadi dasar untuk proses pengolahan dan modifikasi dataset pada pertemuan berikutnya.
